# ResNet으로 CIFAR-10 90% 넘기기

**30일 딥러닝 완전 정복 · Day 18** — 강의 본문: [https://muy-bien-hwa.github.io/For-my-Claude/](https://muy-bien-hwa.github.io/For-my-Claude/)

### 이 노트북에서 할 일
- 잔차 연결이 있는 망과 없는 망(plain)을 같은 깊이로 비교한다
- 작은 ResNet을 증강 + OneCycle + 혼합 정밀도로 학습한다 (목표 90%+)
- 첫 층 합성곱 필터를 시각화한다

> ⚙️ **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. 셀은 위에서부터 `Shift+Enter`로 실행합니다.
> 그래프 라벨은 Colab 기본 글꼴 문제를 피하려고 영어로 표기했습니다.

In [ ]:
import random, os, json, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def train_one_epoch(model, loader, loss_fn, opt, scheduler=None, clip=None, scaler=None):
    model.train(); tot, correct, n = 0.0, 0, 0
    for xb, yb in tqdm(loader, leave=False):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=scaler is not None):
            logits = model(xb); loss = loss_fn(logits, yb)
        opt.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            if clip: scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            if clip: torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        if scheduler is not None: scheduler.step()
        tot += loss.item() * len(yb); correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / n, correct / n

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval(); tot, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb); tot += loss_fn(logits, yb).item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / n, correct / n

def save_ckpt(path, model, opt, epoch, best):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch, "best": best}, path)

def load_ckpt(path, model, opt=None):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck["model"])
    if opt is not None: opt.load_state_dict(ck["opt"])
    return ck["epoch"], ck["best"]

def fit(model, train_dl, val_dl, epochs, lr=1e-3, wd=0.05, ckpt="best.pt", patience=None, label_smoothing=0.0, amp=True, sgd=False):
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    if sgd: opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    else:   opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda") if (amp and device == "cuda") else None
    hist = {k: [] for k in ["tr_loss", "tr_acc", "va_loss", "va_acc"]}; best, bad = 0.0, 0
    for ep in range(epochs):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_dl, loss_fn, opt, sched, clip=1.0, scaler=scaler)
        vl, va = evaluate(model, val_dl, nn.CrossEntropyLoss())
        for k, v in zip(hist, [tl, ta, vl, va]): hist[k].append(v)
        print(f"[{ep+1:02d}/{epochs}] train {tl:.4f}/{ta:.3f}  val {vl:.4f}/{va:.3f}  lr {opt.param_groups[0]['lr']:.2e}  {time.time()-t0:.0f}s")
        if va > best: best, bad = va, 0; save_ckpt(ckpt, model, opt, ep, best)
        else:
            bad += 1
            if patience and bad >= patience: print("early stopping"); break
    load_ckpt(ckpt, model)
    return hist

def plot_hist(hist, title=""):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(hist["tr_loss"], label="train"); ax[0].plot(hist["va_loss"], label="val"); ax[0].set_title(f"{title} loss"); ax[0].legend()
    ax[1].plot(hist["tr_acc"], label="train"); ax[1].plot(hist["va_acc"], label="val"); ax[1].set_title(f"{title} accuracy"); ax[1].legend()
    plt.show()
print("template ready on", device)

## 1. CIFAR-10 데이터와 증강

In [ ]:
mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_tf = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
                               transforms.ToTensor(), transforms.Normalize(mean, std)])
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
full_train = datasets.CIFAR10("data", train=True, download=True, transform=train_tf)
full_val = datasets.CIFAR10("data", train=True, download=True, transform=test_tf)     # 검증은 증강 없이
test_ds = datasets.CIFAR10("data", train=False, download=True, transform=test_tf)
idx = torch.randperm(50000, generator=torch.Generator().manual_seed(0))
train_ds = torch.utils.data.Subset(full_train, idx[:45000]); val_ds = torch.utils.data.Subset(full_val, idx[45000:])
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=1000, num_workers=2); test_dl = DataLoader(test_ds, batch_size=1000, num_workers=2)
classes = full_train.classes; print(classes)

## 2. ResNet (잔차 연결 on/off 가능)

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=1, use_residual=True):
        super().__init__()
        self.use_residual = use_residual
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False); self.bn2 = nn.BatchNorm2d(c_out)
        self.shortcut = nn.Identity()
        if stride != 1 or c_in != c_out:
            self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))
    def forward(self, x):
        out = self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))
        if self.use_residual: out = out + self.shortcut(x)
        return F.relu(out)

class ResNet(nn.Module):
    def __init__(self, blocks=(2, 2, 2, 2), widths=(64, 128, 256, 512), n_classes=10, use_residual=True):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, widths[0], 3, 1, 1, bias=False), nn.BatchNorm2d(widths[0]), nn.ReLU())
        layers, c = [], widths[0]
        for i, (n, w) in enumerate(zip(blocks, widths)):
            for j in range(n):
                layers.append(BasicBlock(c, w, 2 if (j == 0 and i > 0) else 1, use_residual)); c = w
        self.layers = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, n_classes))
    def forward(self, x): return self.head(self.layers(self.stem(x)))

m = ResNet(); print(f"{sum(p.numel() for p in m.parameters())/1e6:.1f}M params", m(torch.randn(2, 3, 32, 32)).shape)

## 3. 실험: 깊은 plain 망 vs ResNet
블록을 (5,5,5) × 폭 (16,32,64)로 쌓은 **32층** 망입니다(ResNet-32 구조). 같은 조건에서 잔차 연결만 끄고 켜서 8에폭씩 학습합니다. 학습 손실 곡선을 비교하세요.

In [ ]:
results = {}
for res in [False, True]:
    seed_everything(0)
    net = ResNet(blocks=(5, 5, 5), widths=(16, 32, 64), use_residual=res).to(device).to(memory_format=torch.channels_last)
    results["ResNet" if res else "Plain"] = fit(net, train_dl, val_dl, epochs=8, lr=2e-3, ckpt=f"cmp_{res}.pt")
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 3.5))
for k, h in results.items(): plt.plot(h["tr_loss"], label=f"{k} train loss"); plt.plot(h["va_acc"], "--", label=f"{k} val acc")
plt.legend(); plt.title("32-layer: plain vs residual"); plt.show()

## 4. 본 학습: ResNet-18 스타일, 목표 90%+
T4 GPU 기준 약 15~20분. 기다리는 동안 강의 본문 5~7절을 읽으세요.

In [ ]:
seed_everything(42)
model = ResNet().to(device).to(memory_format=torch.channels_last)
hist = fit(model, train_dl, val_dl, epochs=30, lr=0.1, wd=5e-4, sgd=True, ckpt="resnet_cifar.pt", label_smoothing=0.1)
plot_hist(hist, "ResNet CIFAR-10")
print("TEST:", evaluate(model, test_dl, nn.CrossEntropyLoss()))

## 5. 첫 층 필터 시각화
CNN이 스스로 학습한 64개의 3×3 컬러 필터입니다. 색 덩어리, 방향성 있는 경계 검출기가 보이나요?

In [ ]:
w = model.stem[0].weight.detach().cpu()           # (64, 3, 3, 3)
w = (w - w.min()) / (w.max() - w.min())
fig, axs = plt.subplots(4, 16, figsize=(14, 3.6))
for a, f in zip(axs.ravel(), w): a.imshow(f.permute(1, 2, 0)); a.axis("off")
plt.suptitle("learned first-layer filters"); plt.show()

## 6. 과제
- plain 망과 ResNet의 결과를 학습 노트에 기록하세요. 어느 쪽 학습 손실이 더 빨리 줄었나요?
- `blocks=(9,9,9)`(56층)로 바꾸면 plain 망은 어떻게 되나요?